# Panel setup

Prepare the regression panel from knowledge matrices already produced by
`know_calc_loop`. Run this notebook from top to bottom after selecting the run in
`project_settings.py`.

The three stages are:
1. Map the existing matrices and auxiliary files to the common sector classification.
2. Split incoming knowledge into four origin channels and build the knowledge panel.
3. Merge the knowledge panel with STAN data and add the variables used in estimation.

## Before running

- Set `BASE`, `YEARS`, `RD_UNIT`, `DIMENSION`, the technology selection, and the
  shock settings in `project_settings.py`.
- Direct and embodied channels are built together for the physical bases; running
  the full notebook twice just to switch between `dir` and `embod` is unnecessary.
- For CAPITAL, select `METRIC_WANTED = "embod_investment"` to include the investment
  channels alongside direct and embodied supply-chain channels. Their source
  matrices must already exist.
- Keep the STAN intermediate files and `intensite_propre.csv` in
  `code/panel_estimation/Panels_intermediaires/`. In `project_settings.py`, set
  `PANEL_CODE_DIR = PATH_WRK + "code/panel_estimation/"` after moving the code.
- Make the directory containing `project_settings.py` available to the notebook's
  Python kernel so that the import below succeeds.
- For a shock run, prepare the files selected by `AGG_SUFFIX` beforehand: the
  aggregation stage below writes baseline `_agg.csv` files, not shocked matrices.

## Outputs and next step

| Output | Purpose |
| --- | --- |
| `_agg.csv` files beside the source matrices and auxiliary files | Common sector mapping |
| `RESULTS_DIR/know_panels/panel_estimation_{BASE}.csv` | Knowledge channels and available trade/production variables |
| `RESULTS_DIR/regression_panels/panel_{BASE}_intermediaire_F.csv` | Intermediate panel after adding broad sectors C and H |
| `RESULTS_DIR/regression_panels/panel_{BASE}_F.csv` | Final panel, including own R&D intensity |

The notebooks in `code/panel_estimation/estimation_notebooks/` must read the final
file from `regression_panels/`. Re-running this notebook replaces the outputs for
the selected run and base. No regression is estimated here.

## Imports and project settings

Reload the central settings so that changes saved before execution are applied.
The printed paths identify the selected run and shock scenario.

In [21]:
import glob
from pathlib import Path
import os
import pandas as pd
import numpy as np
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, SRC_DIRS, SRC_DIR, METRIC_SPE, YEARS, RESULTS_DIR, RUN_DIR, CHOC, CHOC_SCENARIO, AGG_SUFFIX, METRIC_WANTED, PANEL_CODE_DIR
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO}): matrices *{AGG_SUFFIX}.csv" if CHOC else ": baseline matrices *_agg.csv"))
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO})" if CHOC else "") + f" -> {RESULTS_DIR}")

CHOC = False: baseline matrices *_agg.csv
CHOC = False -> D:/ENTICE/runs/SRDQ (c,s)/


# 1. Aggregate to common mapping

Read `mapping/final_aggregation/map_agg_{BASE}.xlsx` and map the source sectors to
the common classification. The functions below handle MARIO matrices, patent
citations, FDI, vectors, and auxiliary production/trade files according to their
layouts. Sectors missing from a mapping are reported and kept unchanged.

This stage processes the raw files found in `SRC_DIRS`, including available years
outside `YEARS`. Files already bearing an `_agg` suffix are skipped as inputs.

In [22]:
# folders written by know_calc_loop: "SRDV (c,s)" / "SRDQ (c,s)", or one folder per technology of TECHS in (c,s,t)
# (RD_UNIT, DIMENSION and TECHS are set in project_settings.py)
OUT_DIRS = SRC_DIRS
MAP_DIR = PATH_WRK + "mapping/final_aggregation"
BASES = [BASE]
# BASES = ["FIGARO", "EXIOBASE", "CAPITAL", "GTAP", "FDI", "OECD_ICIO", "Patents_Citation"]

MATRIX_PREFIXES = ["rd_dir_CI", "rd_embod_CI", "rd_imp_CI", "rd_tot_CI", "z_CI", "rd_dir_CAPITAL", "rd_embod_CAPITAL",
                   "rd_embod_investment_CAPITAL", "rd_dir_cit", "rd_dir_fdips"]

# Two matrices are not mario-based and need their own parser (see AGGREGATORS below):
# - rd_dir_cit (Patents_Citation): only 2 column levels (Region, Sector) instead of 3 (Region, Level, Item),
#   and an extra "Year" level on the rows (Year, Region, Sector)
# - rd_dir_fdips (FDI): rows = host (Region, Sector), columns = investing country only (1 level, no sector)

# Wide files with Region[, Sector] identifiers and year columns, separated by ";"
# -> a single file (all years as columns), in SRDQ (c,s)/{base}/output_loop_{base}/
LONG_SECTOR_FILES = ["production_{base}.csv", "exports_{base}.csv", "imports_intermediate_{base}.csv"]

# Wide files with no Sector column (Region + year columns only) -> no sector
# aggregation needed, just copied as-is under the _agg name
NO_SECTOR_FILES = ["imports_final_demand_{base}.csv"]


def load_mapping(base):
    p = os.path.join(MAP_DIR, f"map_agg_{base}.xlsx")
    df = pd.read_excel(p, sheet_name="Sector", dtype=str)
    # column "Sector" = the base's own aggregation target (source, e.g. "C20_C21")
    # column "aggregation" = the common final sector (destination, e.g. "C20+C21")
    return dict(zip(df["Sector"], df["aggregation"]))


def warn_unmapped(items, mapping, source):
    # sectors present in the file but missing from the base's map_agg:
    # kept as-is (identity) instead of raising -- the state of the mapping
    # is left to the user to handle.
    missing = sorted(set(items) - set(mapping))
    if missing:
        print(f"  [warning] {os.path.basename(source)}: sectors missing from map_agg (kept as-is) -> {missing}")
    return missing


def apply_mapping(values, mapping):
    return [mapping.get(v, v) for v in values]


def aggregate_matrix_file(path, mapping):
    df = pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])
    df.index.names = df.columns.names = ["Region", "Level", "Item"]
    warn_unmapped(df.index.get_level_values("Item").unique(), mapping, path)

    idx = df.index.to_frame(index=False)
    idx["Item"] = apply_mapping(idx["Item"], mapping)
    df.index = pd.MultiIndex.from_frame(idx)

    cols = df.columns.to_frame(index=False)
    cols["Item"] = apply_mapping(cols["Item"], mapping)
    df.columns = pd.MultiIndex.from_frame(cols)

    df = df.groupby(level=["Region", "Level", "Item"]).sum()               # rows
    df = df.T.groupby(level=["Region", "Level", "Item"]).sum().T           # columns
    return df.sort_index().sort_index(axis=1)


def aggregate_citation_matrix_file(path, mapping):
    # columns: 2 levels (Region, Sector) ; rows: 3 levels (Year, Region, Sector)
    df = pd.read_csv(path, sep=";", header=[0, 1], index_col=[0, 1, 2])
    df.index.names = ["Year", "Region", "Sector"]
    df.columns.names = ["Region", "Sector"]
    warn_unmapped(df.index.get_level_values("Sector").unique(), mapping, path)

    idx = df.index.to_frame(index=False)
    idx["Sector"] = apply_mapping(idx["Sector"], mapping)
    df.index = pd.MultiIndex.from_frame(idx)

    cols = df.columns.to_frame(index=False)
    cols["Sector"] = apply_mapping(cols["Sector"], mapping)
    df.columns = pd.MultiIndex.from_frame(cols)

    df = df.groupby(level=["Year", "Region", "Sector"]).sum()               # rows
    df = df.T.groupby(level=["Region", "Sector"]).sum().T                   # columns
    return df.sort_index().sort_index(axis=1)


def aggregate_fdi_matrix_file(path, mapping):
    # rows: 2 levels (Region, Sector) = host region and sector ; columns: 1 level = investing country (no sector level:
    # only the sectors of the rows are aggregated)
    df = pd.read_csv(path, sep=";", index_col=[0, 1])
    df.index.names = ["Region", "Sector"]
    warn_unmapped(df.index.get_level_values("Sector").unique(), mapping, path)

    idx = df.index.to_frame(index=False)
    idx["Sector"] = apply_mapping(idx["Sector"], mapping)
    df.index = pd.MultiIndex.from_frame(idx)

    df = df.groupby(level=["Region", "Sector"]).sum()                       # rows
    return df.sort_index().sort_index(axis=1)


# parser of each matrix; every other prefix uses aggregate_matrix_file (mario-based matrices)
AGGREGATORS = {"rd_dir_cit": aggregate_citation_matrix_file, "rd_dir_fdips": aggregate_fdi_matrix_file}


def aggregate_vector_file(path, mapping):
    # vector_{metric}_{base}_{year}.csv (OUTPUT_TYPE "vector" of the main program): one row per destination, last column = value.
    # Keys (Region, Level, Item) for the mario-based bases, (Region, Sector) for Patents_Citation, Region only for FDI
    # (no sector: copied as-is)
    df = pd.read_csv(path, sep=";", dtype={"Region": str, "Level": str, "Item": str, "Sector": str})
    keys, value = list(df.columns[:-1]), df.columns[-1]
    sector = next((k for k in ("Item", "Sector") if k in keys), None)
    if sector is None:
        return df
    warn_unmapped(df[sector].unique(), mapping, path)
    df[sector] = apply_mapping(df[sector], mapping)
    return df.groupby(keys, as_index=False)[value].sum().sort_values(keys)


def aggregate_long_file(path, mapping):
    df = pd.read_csv(path, sep=";", dtype={"Region": str, "Sector": str})
    warn_unmapped(df["Sector"].unique(), mapping, path)
    df["Sector"] = apply_mapping(df["Sector"], mapping)
    year_cols = [c for c in df.columns if c not in ("Region", "Sector")]
    return df.groupby(["Region", "Sector"], as_index=False)[year_cols].sum()


def out_path(path):
    root, ext = os.path.splitext(path)
    return root + "_agg" + ext


def is_already_aggregated(path):
    return "_agg" in os.path.basename(path)   # _agg.csv and _agg_choc.csv are results of an aggregation


def run_base(base, out_dir):
    mapping = load_mapping(base)
    done = 0
    base_dir = os.path.join(out_dir, base)
    if not os.path.isdir(base_dir):
        print(f"[{base}] folder missing, skipped: {base_dir}")
        return

    for year_dir in sorted(glob.glob(os.path.join(base_dir, f"output_loop_{base}_*"))):
        for prefix in MATRIX_PREFIXES:
            aggregator = AGGREGATORS.get(prefix, aggregate_matrix_file)
            for f in glob.glob(os.path.join(year_dir, f"{prefix}_{base}_*.csv")):
                if is_already_aggregated(f):
                    continue
                aggregator(f, mapping).to_csv(out_path(f), sep=";")
                done += 1
            for f in glob.glob(os.path.join(year_dir, f"vector_{prefix}_{base}_*.csv")):
                if is_already_aggregated(f):
                    continue
                aggregate_vector_file(f, mapping).to_csv(out_path(f), sep=";", index=False)
                done += 1

    no_year_dir = os.path.join(base_dir, f"output_loop_{base}")
    if os.path.isdir(no_year_dir):
        for pattern in LONG_SECTOR_FILES:
            f = os.path.join(no_year_dir, pattern.format(base=base))
            if os.path.exists(f) and not is_already_aggregated(f):
                aggregate_long_file(f, mapping).to_csv(out_path(f), sep=";", index=False)
                done += 1

        for pattern in NO_SECTOR_FILES:
            f = os.path.join(no_year_dir, pattern.format(base=base))
            if os.path.exists(f) and not is_already_aggregated(f):
                pd.read_csv(f, sep=";").to_csv(out_path(f), sep=";", index=False)
                done += 1

    print(f"[{base}] _agg files created: {done} ({out_dir})")


### Run aggregation

Execute the aggregation functions for each configured source directory and base.


In [23]:
for out_dir in OUT_DIRS:
    for b in BASES:
        try:
            run_base(b, out_dir)
        except Exception as e:
            print(f"[{b}] ERROR ({out_dir}): {e}")

[EXIOBASE] _agg files created: 284 (D:/ENTICE/SRDQ (c,s)/)


# 2. Compute knowledge blocks

Read the mapped knowledge matrices for the selected run, base, and years. Matrix
rows identify the origin of knowledge; columns identify the receiving activity.
The resulting panel has one observation per destination region, sector, and year.

### Load a knowledge matrix

Use the matrix layout appropriate to the base and remove extra index levels so
that both axes are labelled `(Region, Sector)`. `AGG_SUFFIX` selects the baseline
or already-prepared shock matrices. The FDI layout requires separate treatment
before it can be used as a region-by-sector destination matrix in this stage.

In [24]:
def load_your_matrix(metric: str, year: int, base: str) -> pd.DataFrame:
    """
    Load a knowledge matrix exported by know_calc_loop and mapped to common sectors.
    Remove the MARIO Level or patent Year level, leaving Region and Sector on both axes.
    """

    phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

    file_path = Path(SRC_DIR + f"{BASE}/output_loop_{BASE}_{year}/{metric}_{base}_{year}{AGG_SUFFIX}.csv")   # SRC_DIR: folder of the run chosen in project_settings; AGG_SUFFIX: _agg (baseline) or _agg_choc (CHOC = True)

    if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
    if base in phys:
        matrix = pd.read_csv(
            file_path,
            sep=";",
            header=[0, 1, 2],
            index_col=[0, 1, 2],
        )

        matrix.index = matrix.index.droplevel("Level")
        matrix.columns = matrix.columns.droplevel("Level")
    else:
        matrix = pd.read_csv(
            file_path,
            sep=";",
            header=[0, 1],
            index_col=[0, 1, 2],
        )

        if "Year" in matrix.index.names or matrix.index.nlevels == 3:
            matrix.index = matrix.index.droplevel(0)

    matrix.index.names = ["Region", "Sector"]
    matrix.columns.names = ["Region", "Sector"]

    return matrix

### Define the four origin channels

For each destination, sum incoming knowledge over four mutually exclusive groups:
same sector/same country, same sector/other country, other sector/same country,
and other sector/other country. Their sum equals the destination's total incoming
knowledge.

In [25]:
def compute_4_blocks(matrix: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Matrix rows identify origins (Region, Sector); columns identify destinations.

    Return one row per destination (Region, Sector), with four columns that
    partition all incoming knowledge by origin country and sector.
    """
    regions_rows = matrix.index.get_level_values("Region").to_numpy()
    sectors_rows = matrix.index.get_level_values("Sector").to_numpy()
    regions_cols = matrix.columns.get_level_values("Region").to_numpy()
    sectors_cols = matrix.columns.get_level_values("Sector").to_numpy()

    same_region = regions_rows[:, None] == regions_cols[None, :]
    same_sector = sectors_rows[:, None] == sectors_cols[None, :]

    values = matrix.to_numpy()

    def block_sum(mask):
        return pd.Series(np.where(mask, values, 0.0).sum(axis=0), index=matrix.columns)

    out = pd.DataFrame({
        f"rd_same_sector_same_country_{metric}": block_sum(same_sector & same_region),
        f"rd_same_sector_diff_country_{metric}": block_sum(same_sector & ~same_region),
        f"rd_diff_sector_same_country_{metric}": block_sum(~same_sector & same_region),
        f"rd_diff_sector_diff_country_{metric}": block_sum(~same_sector & ~same_region),
    })
    out.index.names = ["Region", "Sector"]
    return out

### Define the trade and production merge

Read the mapped auxiliary files, convert year columns to long format, and join
exports, imports, and production to each destination observation. These variables
are added for the physical bases.

In [26]:
def add_trade_and_production(path_wrk: str, year: list, base: str, df: pd.DataFrame) -> pd.DataFrame:
    """Merge mapped trade and production series into the knowledge panel."""
    def load_and_melt(name, value_name, id_vars):
        path = SRC_DIR + f"{base}/output_loop_{base}/{name}_{base}_agg.csv"
        wide = pd.read_csv(path, header=0, sep=";")
        year_cols = [c for c in wide.columns if c not in id_vars]
        long = wide.melt(
            id_vars=id_vars,
            value_vars=year_cols,
            var_name="year",
            value_name=value_name,
        )
        long["year"] = long["year"].astype(int)
        return long

    exports = load_and_melt("exports", "exports", ["Region", "Sector"])
    imports_final_demand = load_and_melt("imports_final_demand", "imports_final_demand", ["Region"])
    imports_intermediate = load_and_melt("imports_intermediate", "imports_intermediate", ["Region", "Sector"])
    production = load_and_melt("production", "production", ["Region", "Sector"])

    df_complete = df.reset_index()

    for y in year:
        if y not in exports["year"].values:
            raise AssertionError(f"Year {y} not in {base}_exports")
        if y not in imports_final_demand["year"].values:
            raise AssertionError(f"Year {y} not in {base}_imports_final_demand")
        if y not in imports_intermediate["year"].values:
            raise AssertionError(f"Year {y} not in {base}_imports_intermediate")
        if y not in production["year"].values:
            raise AssertionError(f"Year {y} not in {base}_production")

    df_complete = df_complete.merge(exports, on=["Region", "Sector", "year"], how="left")
    df_complete = df_complete.merge(imports_final_demand, on=["Region", "year"], how="left")
    df_complete = df_complete.merge(imports_intermediate, on=["Region", "Sector", "year"], how="left")
    df_complete = df_complete.merge(production, on=["Region", "Sector", "year"], how="left")

    df_complete = df_complete[df_complete["year"].isin(year)]
    df_complete = df_complete.set_index(["Region", "Sector", "year"])

    return df_complete

### Build and save the knowledge panel

Compute the channels for each year, combine direct and embodied measures where
available, and add CAPITAL investment channels when selected. Save the result in
`know_panels/`; the next stage reads this file.

In [27]:
METRIC = METRIC_SPE[f"{BASE}_dir"]
panel_rows = []
for year in YEARS:
    matrix = load_your_matrix(METRIC, year, BASE)
    df_year = compute_4_blocks(matrix, METRIC)
    df_year["year"] = year
    panel_rows.append(df_year.reset_index())
panel = pd.concat(panel_rows, ignore_index=True).set_index(["Region", "Sector", "year"])

if BASE not in ["Patents_Citation", "FDI"]:
    METRIC = METRIC_SPE[f"{BASE}_embod"]
    panel_rows2 = []
    for year in YEARS:
        matrix2 = load_your_matrix(METRIC, year, BASE)
        df_year2 = compute_4_blocks(matrix2, METRIC)
        df_year2["year"] = year
        panel_rows2.append(df_year2.reset_index())
    panel2 = pd.concat(panel_rows2, ignore_index=True).set_index(["Region", "Sector", "year"])

    panel = pd.concat([panel, panel2], axis=1)

    if BASE == "CAPITAL" and METRIC_WANTED == "embod_investment":
        metric_investment = METRIC_SPE["CAPITAL_embod_investment"]
        panel_rows_investment = []

        for year in YEARS:
            matrix_investment = load_your_matrix(
                metric_investment, year, BASE
            )
            df_investment = compute_4_blocks(
                matrix_investment, metric_investment
            )
            df_investment["year"] = year
            panel_rows_investment.append(df_investment.reset_index())

        panel_investment = pd.concat(
            panel_rows_investment, ignore_index=True
        ).set_index(["Region", "Sector", "year"])

        panel = pd.concat([panel, panel_investment], axis=1)

    panel = add_trade_and_production(PATH_WRK, YEARS, BASE, panel)

os.makedirs(RESULTS_DIR + "know_panels", exist_ok=True)
panel.to_csv(RESULTS_DIR + f"know_panels/panel_estimation_{BASE}.csv")
print(f"run {RUN_DIR} | {BASE} | years {YEARS} -> {RESULTS_DIR}know_panels/panel_estimation_{BASE}.csv")

run SRDQ (c,s) | EXIOBASE | years [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022] -> D:/ENTICE/runs/SRDQ (c,s)/know_panels/panel_estimation_EXIOBASE.csv


# 3. Build the final regression panel

Add broad sectors C and H, merge with the STAN cost and factor-price data, and
construct the knowledge aggregates, the 2009 indicator, production concentration,
and R&D intensity measures. The final export includes all these variables and
does not split countries into low- and high-intensity groups.

### Read and validate the knowledge panel

Reload the file just produced. If CAPITAL investment is selected, check that all
four investment channels are present before continuing.

In [28]:
panel_file = RESULTS_DIR + f"know_panels/panel_estimation_{BASE}.csv"
print("Reading knowledge panel:", panel_file)
df = pd.read_csv(panel_file)

# The investment channels must already be present in the knowledge panel created above.
if BASE == "CAPITAL" and METRIC_WANTED == "embod_investment":
    investment_metric = METRIC_SPE["CAPITAL_embod_investment"]
    investment_columns = [
        f"rd_same_sector_same_country_{investment_metric}",
        f"rd_same_sector_diff_country_{investment_metric}",
        f"rd_diff_sector_same_country_{investment_metric}",
        f"rd_diff_sector_diff_country_{investment_metric}",
    ]
    missing = [c for c in investment_columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"Investment columns missing from the knowledge panel: {missing}. "
            "Re-run stage 2 with BASE='CAPITAL' and "
            "METRIC_WANTED='embod_investment', using the same RESULTS_DIR. "
            f"Expected file: {panel_file}"
        )

Reading knowledge panel: D:/ENTICE/runs/SRDQ (c,s)/know_panels/panel_estimation_EXIOBASE.csv


In [29]:
print(df.columns)

Index(['Region', 'Sector', 'year', 'rd_same_sector_same_country_rd_dir_CI',
       'rd_same_sector_diff_country_rd_dir_CI',
       'rd_diff_sector_same_country_rd_dir_CI',
       'rd_diff_sector_diff_country_rd_dir_CI',
       'rd_same_sector_same_country_rd_embod_CI',
       'rd_same_sector_diff_country_rd_embod_CI',
       'rd_diff_sector_same_country_rd_embod_CI',
       'rd_diff_sector_diff_country_rd_embod_CI', 'exports',
       'imports_final_demand', 'imports_intermediate', 'production'],
      dtype='str')


In [30]:
df = df.rename(columns={"year": "annee"})

### Add broad sectors C and H

Sum the selected additive columns by region and year across C subsectors to build
manufacturing (C). For physical bases, apply the same procedure to the configured
H subsectors. Keep the detailed sectors alongside these broad aggregates.

In [31]:
c_subsectors = df.loc[df['Sector'].str.startswith('C', na=False), 'Sector'].unique().tolist()

mask_c = df['Sector'].isin(c_subsectors)

phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

if BASE in phys:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',

            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',

            'exports',
            'imports_final_demand',
            'imports_intermediate',
            'production']
else:
    col = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}']

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    col += ["rd_same_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_same_sector_diff_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_same_country_rd_embod_investment_CAPITAL",
            "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL",]

c_agg = (
    df[mask_c]
    .groupby(['Region', 'annee'], as_index=False)
    [col]
    .sum()
)
c_agg['Sector'] = 'C'

df_final = pd.concat([df, c_agg], ignore_index=True)

In [32]:
if BASE in phys:
    h_subsectors = df.loc[df['Sector'].str.startswith('H', na=False), 'Sector'].unique().tolist()
    mask_h = df_final['Sector'].isin(h_subsectors)

    h_agg = (
        df_final[mask_h]
        .groupby(['Region', 'annee'], as_index=False)
        [col]
        .sum()
    )
    h_agg['Sector'] = 'H'

    df_final1 = pd.concat([df_final, h_agg], ignore_index=True)
else:
    df_final1 = df_final

### Save the intermediate sector panel

Store the panel with broad sectors before the STAN merge. This file is an
intermediate output; estimations use the final `panel_{BASE}_F.csv` file.

In [33]:
os.makedirs(RESULTS_DIR + 'regression_panels', exist_ok=True)
df_final1.to_csv(RESULTS_DIR + f'regression_panels/panel_{BASE}_intermediaire_F.csv')

In [34]:
df_final1

,Region,Sector,annee,rd_same_sector_same_country_rd_dir_CI,rd_same_sector_diff_country_rd_dir_CI,rd_diff_sector_same_country_rd_dir_CI,rd_diff_sector_diff_country_rd_dir_CI,rd_same_sector_same_country_rd_embod_CI,rd_same_sector_diff_country_rd_embod_CI,rd_diff_sector_same_country_rd_embod_CI,rd_diff_sector_diff_country_rd_embod_CI,exports,imports_final_demand,imports_intermediate,production
0,AT,A,1995,0.266540,0.384750,52.944931,47.626371,0.350535,0.723019,160.453827,157.975087,181.963870,1.486011e+04,908.345668,8.794762e+03
1,AT,B,1995,0.375862,0.186910,17.793596,5.743556,0.448084,0.335228,41.883169,22.634096,56.363108,1.486011e+04,100.182006,1.136038e+03
2,AT,C10_C12,1995,1.501903,2.333629,86.017330,27.186100,3.500180,3.942197,290.114641,187.894080,587.010739,1.486011e+04,930.614479,1.005106e+04
3,AT,C13_C15,1995,1.432178,3.514206,21.756333,6.660152,1.541429,4.144766,49.885576,31.266826,922.116132,1.486011e+04,163.445058,1.737351e+03
4,AT,C16_C18,1995,24.756533,4.952038,101.757479,22.669687,32.939959,7.742200,345.204919,195.508223,1935.597272,1.486011e+04,973.615961,1.098599e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42331,US,H,2018,17203.767843,290.955902,26422.013077,2208.158002,22279.481208,525.367411,45660.397672,7747.007226,94730.399206,5.471909e+06,61229.908341,3.103204e+06
42332,US,H,2019,19949.475052,259.208049,33304.861700,2392.374671,26907.962664,510.175395,57099.912614,8574.263887,103810.094507,6.024783e+06,67532.491401,3.367028e+06
42333,US,H,2020,25093.425453,254.792172,27343.913547,2271.497946,31729.320788,518.252463,44130.255132,7293.761182,92062.320500,5.291099e+06,56582.772874,3.359561e+06
42334,US,H,2021,25954.799419,388.634589,38045.886580,2363.191433,34329.791131,759.477342,61751.150495,8415.676212,97560.478023,6.064946e+06,63618.490914,3.592476e+06


### Merge with STAN data

Read `panel_va_STAN.csv` for physical bases or `panel_cit1.csv` for the other bases,
then merge by region, sector, and year. STAN observations form the left side of
the merge. For physical bases, keep the source database's production separately
as `production_{BASE}`.

In [35]:
if BASE in phys:
    va_STAN = pd.read_csv(PATH_WRK + 'code/panel_estimation/Panels_intermediaires/panel_va_STAN.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]

    df_final1 = df_final1.rename(columns={"production": f"production_{BASE}"})
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])
else:
    va_STAN = pd.read_csv(PATH_WRK + 'code/panel_estimation/Panels_intermediaires/panel_cit1.csv')
    va_STAN = va_STAN[va_STAN["annee"].isin(YEARS)]
    va_STAN = va_STAN[va_STAN["Sector"].isin(df_final1["Sector"])]
    df = va_STAN.merge(df_final1, how='left', on=['Region', 'Sector', 'annee'])
    df = df.drop(columns=["Unnamed: 0"])

df

,Region,Sector,annee,VA_V,VA_Q,VA_costFactors,comp_emp,emp,net_stock_capital,production,...,rd_diff_sector_same_country_rd_dir_CI,rd_diff_sector_diff_country_rd_dir_CI,rd_same_sector_same_country_rd_embod_CI,rd_same_sector_diff_country_rd_embod_CI,rd_diff_sector_same_country_rd_embod_CI,rd_diff_sector_diff_country_rd_embod_CI,exports,imports_final_demand,imports_intermediate,production_EXIOBASE
0,AT,A,1995,3843.823732,3410.753099,5023.693732,348.500000,38.626784,3.428939e+04,7664.729833,...,52.944931,47.626371,0.350535,0.723019,160.453827,157.975087,181.963870,1.486011e+04,908.345668,8.794762e+03
1,AT,A,1996,3588.173927,3354.544562,4309.291927,354.200000,38.047075,3.438198e+04,7436.299892,...,70.066918,49.836471,0.500391,1.833288,126.647567,168.895495,214.838322,1.727854e+04,880.684899,8.202025e+03
2,AT,A,1997,3693.656888,3478.446066,4251.222888,364.700000,38.474740,3.479585e+04,7714.325937,...,76.368708,44.577097,0.599020,1.604589,135.019978,160.350111,236.581217,1.936133e+04,932.554859,8.348596e+03
3,AT,A,1998,3590.057497,3625.242586,4249.225497,366.000000,37.816498,3.498068e+04,7467.688729,...,63.272168,46.877283,0.642565,1.675288,198.281470,166.326670,295.848279,2.087913e+04,919.797470,8.043761e+03
4,AT,A,1999,3616.638040,3843.736884,4315.897040,382.300000,39.327879,3.516822e+04,7480.581422,...,64.175994,49.955113,0.678716,4.519212,140.820124,169.253678,310.963809,2.235032e+04,959.229835,8.198995e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15199,US,R+S+T,2015,403824.172649,433197.497956,393560.340571,267416.971634,10318.000000,1.022369e+06,670507.180879,...,14141.993728,379.086772,128.006084,5.433738,27048.531198,2396.716349,27650.469925,1.033579e+06,14280.929361,9.130389e+05
15200,US,R+S+T,2016,420530.767816,438314.944498,409734.759237,278968.564174,10471.000000,1.047762e+06,701452.164244,...,16102.061388,395.225320,146.906787,4.564936,31152.450601,2638.990386,27263.110241,1.037830e+06,13930.696062,9.776044e+05
15201,US,R+S+T,2017,435249.731220,449286.366872,423538.087025,289233.014358,10519.000000,1.062726e+06,727146.899842,...,17192.452477,394.565838,168.724233,6.582886,32844.064388,2600.966407,25652.049924,1.082975e+06,14401.154481,9.668593e+05
15202,US,R+S+T,2018,457138.704255,465631.493211,444723.251620,304570.694274,10767.000000,1.082054e+06,763918.279636,...,19712.094039,460.908029,194.159984,5.758728,38157.101032,2933.870238,27400.950825,1.094382e+06,15193.418535,9.958279e+05


### Construct knowledge aggregates and controls

For each available metric, combine the four channels into domestic (D), foreign
(F), same-sector (I), other-sector (X), and total knowledge. Add the 2009 indicator
and an HHI based on countries' production shares within each sector and year.
For CAPITAL, `embod` denotes supply-chain knowledge and `embod_investment` denotes
investment knowledge.

In [36]:
# Embodied knowledge; for CAPITAL, this is the supply-chain measure.
if BASE in phys:
    col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_F'] = df[col_F].sum(axis=1)
    df['rd_embod_D'] = df[col_D].sum(axis=1)

    col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            ]
    col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod_I'] = df[col_I].sum(axis=1)
    df['rd_embod_X'] = df[col_X].sum(axis=1)

    col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_embod"]}',
            f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_embod"]}'
            ]
    df['rd_embod'] = df[col_S].sum(axis=1)

# Direct knowledge.
col_F = [f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_D = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_F'] = df[col_F].sum(axis=1)
df['rd_dir_D'] = df[col_D].sum(axis=1)

col_I = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        ]
col_X = [f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir_I'] = df[col_I].sum(axis=1)
df['rd_dir_X'] = df[col_X].sum(axis=1)

col_S = [f'rd_same_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_same_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_same_country_{METRIC_SPE[f"{BASE}_dir"]}',
        f'rd_diff_sector_diff_country_{METRIC_SPE[f"{BASE}_dir"]}'
        ]
df['rd_dir'] = df[col_S].sum(axis=1)

df["2009_crisis"]=(df["annee"]==2009).astype(int)

# col = 'production_EXIOBASE_mapF'
# df['share'] = df['production'] / df.groupby(['Sector', 'annee'])[col].transform('sum')
df['share'] = df['production'] / df.groupby(['Sector', 'annee'])['production'].transform('sum')
df['HHI'] = df.groupby(['Sector', 'annee'])['share'].transform(lambda s: (s**2).sum())

if BASE == "CAPITAL" and "rd_same_sector_same_country_rd_embod_investment_CAPITAL" in df.columns:
    same_same = "rd_same_sector_same_country_rd_embod_investment_CAPITAL"
    same_diff = "rd_same_sector_diff_country_rd_embod_investment_CAPITAL"
    diff_same = "rd_diff_sector_same_country_rd_embod_investment_CAPITAL"
    diff_diff = "rd_diff_sector_diff_country_rd_embod_investment_CAPITAL"

    df["rd_embod_investment_D"] = df[[same_same, diff_same]].sum(axis=1)
    df["rd_embod_investment_F"] = df[[same_diff, diff_diff]].sum(axis=1)
    df["rd_embod_investment_I"] = df[[same_same, same_diff]].sum(axis=1)
    df["rd_embod_investment_X"] = df[[diff_same, diff_diff]].sum(axis=1)
    df["rd_embod_investment"] = df[[same_same, same_diff, diff_same, diff_diff]].sum(axis=1)

In [37]:
df = df.drop(columns=['share'])

### Compute accessible-knowledge intensity

Divide the selected accessible-knowledge measure by STAN production. Keep the
existing column name `intensité` for compatibility with the estimation code.
This measure differs from the own R&D expenditure intensity added below.

In [38]:
df['intensité'] = df[f'rd_{METRIC_WANTED}'] / df[f'production']

### Compute own R&D expenditure intensity

Read `intensite_propre.csv`, prepared upstream from own R&D expenditure and STAN
production in current million euros, independently of the knowledge database and
technology. Compute `int_propre = 100 * own R&D expenditure / production` for each
region, sector, and year: a value of 2 means 2%. This variable is used in the
interaction in specification (5). Production values at or below 1 million euros
are excluded from this ratio, following the existing rule.

In [39]:
# Own R&D / production: RDV and PRODV are prepared once by prépa_va_STAN, independently of the base and technology.
# rd_propre is annual R&D expenditure in current EUR million; production is STAN production in current EUR million (annual exchange rates), including C and H.
df_int = (pd.read_csv(PANEL_CODE_DIR + "Panels_intermediaires/intensite_propre.csv")
            .rename(columns={"rd_propre_V": "rd_propre", "production_V": "production"}))

# Own R&D intensity = the sector's R&D expenditure / the sector's production.
df_int["intensité_vraie"] = df_int["rd_propre"] / df_int["production"]

# Annual own R&D intensity R/Y, expressed in percent, for the regression interaction.
df_int.loc[df_int["production"] <= 1.0, "intensité_vraie"] = np.nan   # Exclude near-zero production to avoid extreme intensity ratios.
int_annual = (df_int[["Region", "Sector", "annee", "intensité_vraie"]]
              .rename(columns={"intensité_vraie": "int_propre"}))
int_annual["int_propre"] = 100 * int_annual["int_propre"]

### Merge own intensity and export the final panel

Join `int_propre` by region, sector, and year, then write the complete regression
panel once to `regression_panels/panel_{BASE}_F.csv`. The estimation notebooks can
reuse this file for the reference, fine-tuned, and scaled approaches.

In [40]:
# Complete regression panel, including own R&D intensity.
df = df.merge(int_annual, on=["Region", "Sector", "annee"], how="left")

os.makedirs(RESULTS_DIR + "regression_panels", exist_ok=True)

df.to_csv(RESULTS_DIR + f"regression_panels/panel_{BASE}_F.csv", index=False)